# **Recommendation System**

**Data Preprocessing:**

Load the dataset into a suitable data structure (e.g., pandas DataFrame).

In [49]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [50]:
df = pd.read_csv("/content/anime.csv")

In [51]:
df.shape

(12294, 7)

In [52]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB


In [53]:
df.describe()

,anime_id,rating,members
count,12294.000000,12064.000000,1.229400e+04
mean,14058.221653,6.473902,1.807134e+04
std,11455.294701,1.026746,5.482068e+04
min,1.000000,1.670000,5.000000e+00
25%,3484.250000,5.880000,2.250000e+02
50%,10260.500000,6.570000,1.550000e+03
75%,24794.500000,7.180000,9.437000e+03
max,34527.000000,10.000000,1.013917e+06


In [54]:
df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [55]:
df.isnull().sum()

,0
anime_id,0
name,0
genre,62
type,25
episodes,0
rating,230
members,0


Handle missing values, if any.

In [56]:
df['genre'] = df['genre'].fillna('Unknown')
df['type'] = df['type'].fillna('Unknown')
df['rating'] = df['rating'].fillna(df['rating'].mean())

In [57]:
df.isnull().sum()

,0
anime_id,0
name,0
genre,0
type,0
episodes,0
rating,0
members,0


**Feature Extraction:**

Decide on the features that will be used for computing similarity (e.g., genres, user ratings).

In [58]:
features = df[['genre', 'type', 'rating', 'episodes']]

Convert categorical features into numerical representations if necessary.

In [59]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df['genre'] = le.fit_transform(df['genre'].astype(str))
df['type'] = le.fit_transform(df['type'].astype(str))

In [60]:
df['genre']

,genre
0,2686
1,161
2,534
3,3240
4,534
...,...
12289,2903
12290,2903
12291,2903
12292,2903


In [61]:
df['type']

,type
0,0
1,5
2,5
3,5
4,5
...,...
12289,3
12290,3
12291,3
12292,3


Normalize numerical features if required.

In [62]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

# Convert 'episodes' to numeric, replacing 'Unknown' with 0
df['episodes'] = df['episodes'].replace('Unknown', '0').astype(float)

df[['rating', 'episodes']] = scaler.fit_transform( df[['rating', 'episodes']])

In [63]:
df['episodes']

,episodes
0,0.000550
1,0.035204
2,0.028053
3,0.013201
4,0.028053
...,...
12289,0.000550
12290,0.000550
12291,0.002200
12292,0.000550


In [64]:
df['rating']

,rating
0,0.924370
1,0.911164
2,0.909964
3,0.900360
4,0.899160
...,...
12289,0.297719
12290,0.313325
12291,0.385354
12292,0.397359


**Recommendation System:**

Design a function to recommend anime based on cosine similarity.

In [65]:
from sklearn.metrics.pairwise import cosine_similarity

# Feature matrix
X = df[['genre', 'type', 'rating', 'episodes']]

# Calculate cosine similarity
similarity = cosine_similarity(X)

print(similarity.shape)

(12294, 12294)


In [66]:
def recommend_anime(anime_name, top_n=5):
    idx = df[df['name'] == anime_name].index[0]
    scores = list(enumerate(similarity[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    scores = scores[1:top_n+1]
    for i in scores:
        print(df.iloc[i[0]]['name'], "-", round(i[1], 3))

In [67]:
recommend_anime("Naruto")

Sakigake!! Otokojuku - 1.0
Kidou Senkan Nadesico - 1.0
Shijou Saikyou no Deshi Kenichi - 1.0
Full Metal Panic! - 1.0
Kurogane no Linebarrels Specials - 1.0


In [69]:
thresholds = [0.3, 0.5, 0.7, 0.9]
anime_name = "Naruto"
idx = df[df['name'] == anime_name].index[0]
for threshold in thresholds:
    count = 0
    for score in similarity[idx]:
        if score > threshold:
            count += 1

    print(f"Threshold {threshold}: {count-1} recommendations")

Threshold 0.3: 12233 recommendations
Threshold 0.5: 12229 recommendations
Threshold 0.7: 12227 recommendations
Threshold 0.9: 12216 recommendations


**Interview Questions**

**1. Can you explain the difference between user-based and item-based collaborative filtering?**

**User-Based Collaborative Filtering:**

Finds users with similar interests.
Recommends items liked by similar users.
Similarity is calculated between users.
Slower for large datasets.
User preferences can change frequently.

**Item-based Collaborative Filtering:**

Finds items that are similar to each other.
Recommends items similar to those the user already likes.
Similarity is calculated between items.
Faster and more scalable.
Item similarities are usually more stable.

**2. What is collaborative filtering, and how does it work?**

Collaborative Filtering is a recommendation technique that suggests items to users based on the preferences and behavior of other users.
It assumes that users who had similar interests in the past will likely have similar interests in the future.

How It Works:
Collect user ratings or interactions with items.
Find similarities between users or items.
Predict which items a user may like.
Recommend those items to the user.

